# 🎬 Recap Dubbing — easy version (v2.1)

**Flow:** Video ingest → Whisper Turbo GPU transcription → Gemini translates the *text only* into natural spoken Myanmar → Edge-TTS → Pydub timeline mix → GPU render (NVENC + flip + color + pure voice) → 9:16 Shorts + Telegram send.

**Why Whisper-first?** Whisper hears the actual speech and gives exact words + timestamps, so there are no missed lines, no wrong timestamps and no invented dialogue. Gemini then translates from exact text instead of guessing from video frames.

## How to use (no code typing needed)
1. **Runtime → Change runtime type → T4 GPU**, then connect.
2. Left panel 🔑 **Secrets**: add `GEMINI_API_KEY` (from Google AI Studio). Optional: `TG_BOT_TOKEN` + `TG_CHAT_ID` for Telegram sending. Never paste keys in code or chat.
3. In the **form cell** below, pick from the dropdowns/checkboxes (YouTube link OR upload — not both).
4. **Runtime → Run all**, wait, then download the mp4s from the download cell.
5. **Phone-friendly UI (optional):** run the Gradio cell at the very end — it prints a public link you can open from your phone. Keep this Colab tab open while it runs; the link dies when the session ends.
6. **Download outputs before the session ends** — Colab runtimes are temporary (or set `SAVE_TO_DRIVE` on in the form).


In [ ]:
import subprocess, sys

print("=== install ===")
subprocess.run(["pip", "install", "-q", "faster-whisper", "google-genai",
                "edge-tts", "pydub", "yt-dlp", "requests", "gradio"], check=True)
# v2.1 fix: faster-whisper calls av.open(..., metadata_errors=...) but an
# older PyAV lacks that argument -> TypeError at transcribe time. Force both
# to their latest compatible versions together.
subprocess.run(["pip", "install", "-q", "-U", "av", "faster-whisper"], check=True)
import av
print("av version:", av.__version__)
print("=== pipeline module ===")
subprocess.run(["wget", "-q",
                "https://raw.githubusercontent.com/htinkyawzaw2017-maker/recap-studio-streamlit/main/colab/recap_pipe.py",
                "-O", "/content/recap_pipe.py"], check=True)
sys.path.insert(0, "/content")
import recap_pipe as rp
print("=== GPU ===")
subprocess.run(["nvidia-smi", "-L"])
import torch
print("torch cuda available:", torch.cuda.is_available())
print("video encoder:", rp.pick_encoder(), "(NVENC falls back to libx264 automatically if no GPU)")


In [ ]:
# ── PICK FROM THE FORM — no code typing needed ─────────────────────
VIDEO_SOURCE = "youtube"  #@param ["youtube", "upload"]
YOUTUBE_URL = ""  #@param {type:"string"}
LANGUAGE = "auto"  #@param ["auto", "zh", "en", "my"]
WHISPER_MODEL = "large-v3-turbo"  #@param ["large-v3-turbo", "medium", "small"]
GEMINI_MODEL = "gemini-2.5-flash-lite"  #@param {type:"string"}
VOICE = "my-MM-NilarNeural"  #@param ["my-MM-NilarNeural", "my-MM-ThihaNeural"]
FLIP = True  #@param {type:"boolean"}
COLOR_TWEAK = True  #@param {type:"boolean"}
MUTE_ORIGINAL = True  #@param {type:"boolean"}
MAKE_SHORTS_9x16 = True  #@param {type:"boolean"}
SEND_TELEGRAM = False  #@param {type:"boolean"}
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
OUTPUT_NAME = "recap_dubbed"  #@param {type:"string"}
# "auto" means Whisper detects the language itself.
LANGUAGE = None if LANGUAGE == "auto" else LANGUAGE
print("source:", VIDEO_SOURCE, "| voice:", VOICE, "| shorts:", MAKE_SHORTS_9x16)


In [ ]:
import json, os

WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)
if VIDEO_SOURCE == "upload":
    from google.colab import files
    up = files.upload()
    SRC = os.path.join(WORK, list(up.keys())[0])
else:
    assert YOUTUBE_URL and YOUTUBE_URL.startswith("http"), "Paste a real YouTube link in the form, or switch VIDEO_SOURCE to upload."
    SRC = rp.download_youtube(YOUTUBE_URL, WORK)
DURATION = rp.probe_duration(SRC)
print(f"video: {SRC}  duration: {DURATION:.1f}s")


In [ ]:
AUDIO = rp.extract_audio(SRC, WORK)
print("audio ready:", AUDIO)


In [ ]:
TRANSCRIPT, detected, device = rp.transcribe(AUDIO, WHISPER_MODEL, LANGUAGE)
json.dump(TRANSCRIPT, open(os.path.join(WORK, "transcript.json"), "w"),
          ensure_ascii=False, indent=1)
print(f"whisper ({device}, detected {detected}): {len(TRANSCRIPT)} segments")
for s in TRANSCRIPT[:5]:
    print(f"{s['start']:>7.2f}  {s['text']}")


In [ ]:
from google.colab import userdata

try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = None
assert GEMINI_API_KEY, "Add GEMINI_API_KEY in Colab Secrets (left panel key icon) — never paste keys in code or chat."
DUB = rp.translate(TRANSCRIPT, GEMINI_API_KEY, GEMINI_MODEL)
json.dump(DUB, open(os.path.join(WORK, "dub_script.json"), "w"),
          ensure_ascii=False, indent=1)
print(f"dub lines: {len(DUB)}")
for d in DUB[:5]:
    print(f"{d['start']:>7.2f}  {d['text']}")


In [ ]:
CLIPS = os.path.join(WORK, "clips")
failed = rp.synthesize(DUB, VOICE, CLIPS)
print(f"clips ok: {len(DUB) - len(failed)} / {len(DUB)}")
for f in failed:
    print("FAILED:", f)


In [ ]:
DUB_TRACK, SRT_PATH, n_srt = rp.mix_and_srt(
    DUB, CLIPS, DURATION, SRC, MUTE_ORIGINAL, WORK)
print("dub track:", DUB_TRACK, "| srt lines:", n_srt)


In [ ]:
OUT, venc = rp.render_video(SRC, DUB_TRACK, DURATION,
                              os.path.join(WORK, OUTPUT_NAME + ".mp4"),
                              FLIP, COLOR_TWEAK)
print("done:", OUT, f"({os.path.getsize(OUT) / 1e6:.1f} MB, encoder {venc})")


In [ ]:
if MAKE_SHORTS_9x16:
    SHORTS = rp.make_shorts(SRC, DUB_TRACK, DURATION,
                            os.path.join(WORK, OUTPUT_NAME + "_9x16.mp4"))
    print("shorts:", SHORTS, f"({os.path.getsize(SHORTS) / 1e6:.1f} MB)")
else:
    SHORTS = None
    print("9:16 skipped")


In [ ]:
from google.colab import files

for f in [OUT] + ([SHORTS] if SHORTS else []) + \
         [os.path.join(WORK, "dub_script.json"), os.path.join(WORK, "dub.srt")]:
    if f and os.path.exists(f):
        files.download(f)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    import shutil
    for f in [OUT] + ([SHORTS] if SHORTS else []):
        if f and os.path.exists(f):
            shutil.copy(f, "/content/drive/MyDrive/")
    print("saved to Drive")


In [ ]:
if SEND_TELEGRAM:
    try:
        tok = userdata.get("TG_BOT_TOKEN")
        chat = userdata.get("TG_CHAT_ID")
    except Exception:
        tok = chat = None
    assert tok and chat, "Add TG_BOT_TOKEN and TG_CHAT_ID in Colab Secrets first."
    import requests
    for f in [OUT] + ([SHORTS] if SHORTS else []):
        if f and os.path.exists(f) and os.path.getsize(f) < 50 * 1024 * 1024:
            with open(f, "rb") as fh:
                r = requests.post(f"https://api.telegram.org/bot{tok}/sendVideo",
                                  data={"chat_id": chat}, files={"video": fh},
                                  timeout=300)
            print(f, r.status_code, r.text[:200])
        else:
            print("skip (>50MB Bot API limit or missing):", f)
else:
    print("Telegram send off (SEND_TELEGRAM=False in the form)")


## 📱 Phone-friendly UI (optional — run this cell only if you want a link)
This cell starts a small web UI and prints a public link. Open that link from your phone: upload a video (or paste a YouTube URL), pick a voice, press **Run dubbing**, and download the finished videos from the page. Notes:
- The cell **keeps running** while the UI is up — that is normal. Stop the cell when you are done.
- Keep this Colab tab open while a job runs (Pro has no background execution).
- The link dies when the session ends — run the cell again for a fresh link.


In [ ]:
import gradio as gr
from google.colab import userdata
import recap_pipe as rp

def ui_run(video_file, youtube_url, voice, make_shorts):
    try:
        api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        api_key = None
    main, shorts, log = rp.run_all(
        video_file=video_file, youtube_url=youtube_url, voice=voice,
        make_shorts_flag=make_shorts, api_key=api_key)
    return main, shorts, log

with gr.Blocks(title="Recap Dubbing") as demo:
    gr.Markdown("Upload a video OR paste a YouTube URL, pick a voice, press Run dubbing.")
    video_file = gr.File(label="Video file (leave empty if using a URL)")
    youtube_url = gr.Textbox(label="YouTube URL (optional)")
    voice = gr.Dropdown(["my-MM-NilarNeural", "my-MM-ThihaNeural"],
                        value="my-MM-NilarNeural", label="Voice")
    make_shorts = gr.Checkbox(value=True, label="Also make 9:16 Shorts")
    btn = gr.Button("Run dubbing")
    out_main = gr.Video(label="Dubbed video")
    out_shorts = gr.Video(label="9:16 Shorts")
    log = gr.Textbox(label="Status")
    btn.click(ui_run, [video_file, youtube_url, voice, make_shorts],
              [out_main, out_shorts, log])
demo.launch(share=True)


## ⚠️ Honest limits (v2)
- **Session is temporary.** Download outputs (or use Drive) before disconnect — Colab does not host anything 24/7.
- **Single voice.** True multi-voice dubbing needs speaker diarization (e.g. pyannote + a Hugging Face token) — a future upgrade, not faked here.
- **Sync is start-anchored.** Each dub clip starts at its Whisper timestamp; if a Myanmar line runs longer than its window it flows into the following gap. The Streamlit app's retime (slowing/freezing video to fit the voice) is the exact-sync solution.
- **Subtitles are an SRT sidecar** (`dub.srt`) — video players shape Myanmar correctly from it. Burned-in PNG subtitles with proper Myanmar shaping (Pillow + Raqm) live in the Streamlit app.
- **Flip/color** reduce automated matches but do **not** guarantee copyright safety.
- **Units:** the run bills the account signed into Colab. AI Pro ≈ 200 compute units/month; if one account runs low, switch accounts (units do not pool). On a machine without an NVIDIA GPU, rendering falls back to libx264 automatically (slower, same output).
